**Task 1 : Big Data Analysis Using Pyspark**

In [ ]:
!pip install pyspark

#Installs PySpark library for handling large-scale (big) data.

In [ ]:
#Initialize Spark Session

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Task1_BigData") \
    .getOrCreate()

#Starts PySpark engine.
#SparkSession is required to process big data.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

# Define schema explicitly to avoid type inference issues
schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("discounted_price", StringType(), True), # Load as string, then clean and cast
    StructField("actual_price", StringType(), True),   # Load as string, then clean and cast
    StructField("discount_percentage", StringType(), True), # Load as string, then clean and cast
    StructField("rating", FloatType(), True),           # Load directly as float
    StructField("rating_count", StringType(), True),   # Load as string, then clean and cast
    StructField("about_product", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("user_name", StringType(), True),
    StructField("review_id", StringType(), True),
    StructField("review_title", StringType(), True),
    StructField("review_content", StringType(), True),
    StructField("img_link", StringType(), True),
    StructField("product_link", StringType(), True)
])

#Load Dataset with explicit schema
df = spark.read.csv("Amazon Sales Dataset.csv", header=True, schema=schema, quote='"', escape='"', multiLine=True)
df.show(5)

#Loads dataset into Spark DataFrame with explicit schema.
#Displays first 5 rows.

+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|product_id|        product_name|            category|discounted_price|actual_price|discount_percentage|rating|rating_count|        about_product|             user_id|           user_name|           review_id|        review_title|      review_content|            img_link|        product_link|
+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|B07JW9H4J1|Wayona Nylon Brai...|Computers&Accesso...|            ₹399|      ₹1,099|                64%|   4.2|      2

In [ ]:
#Check Dataset Structure

df.printSchema()

#Shows column names and data types (important for analysis).

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- discounted_price: string (nullable = true)
 |-- actual_price: string (nullable = true)
 |-- discount_percentage: string (nullable = true)
 |-- rating: float (nullable = true)
 |-- rating_count: string (nullable = true)
 |-- about_product: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_content: string (nullable = true)
 |-- img_link: string (nullable = true)
 |-- product_link: string (nullable = true)



In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType, IntegerType

#Data Cleaning

#1.Remove null values
df = df.dropna()

#2.Remove duplicates
df = df.dropDuplicates()

#3.Clean and Convert numeric columns
from pyspark.sql.functions import col, regexp_replace, when, lit

# 'rating' is now loaded as FloatType directly, so no need to cast here. Drop NaNs if any malformed values got through.
df = df.dropna(subset=["rating"])

# Clean 'discounted_price' and 'actual_price' (remove '₹', commas) and cast to FloatType
df = df.withColumn("discounted_price", regexp_replace(col("discounted_price"), "[₹, ]", ""))
df = df.withColumn("discounted_price", col("discounted_price").cast(FloatType()))

df = df.withColumn("actual_price", regexp_replace(col("actual_price"), "[₹, ]", ""))
df = df.withColumn("actual_price", col("actual_price").cast(FloatType()))

# Clean 'discount_percentage' (remove '%') and cast to FloatType
df = df.withColumn("discount_percentage", regexp_replace(col("discount_percentage"), "%", ""))
df = df.withColumn("discount_percentage", col("discount_percentage").cast(FloatType()))

# Clean 'rating_count' (remove commas) and cast to IntegerType
df = df.withColumn("rating_count", regexp_replace(col("rating_count"), ",", ""))
df = df.withColumn("rating_count", col("rating_count").cast(IntegerType()))

# Drop any rows where these conversions resulted in nulls (optional, but good for robust analysis)
df = df.dropna(subset=["discounted_price", "actual_price", "discount_percentage", "rating_count"])

# Cache the DataFrame to force materialization with correct types
df = df.cache()
# Force cache materialization by performing an action
df.count()

#Cleans the dataset to ensure accurate results.
#Fixes incorrect data types.

1462

In [ ]:
#Basic Analysis
import pyspark.sql.functions as F

#1.Total records
print("Total Records:", df.count())

#2. Summary statistics for 'rating' column (bypassing describe() due to persistent errors)
# The columns 'HelpfulnessNumerator' and 'HelpfulnessDenominator' do not exist in the dataset.
# The column 'Score' should be 'rating'.
print("\nRating Summary Statistics:")
df.agg(
    F.min("rating").alias("min"),
    F.max("rating").alias("max"),
    F.mean("rating").alias("mean"),
    F.stddev("rating").alias("stddev"),
    F.count("rating").alias("count")
).show()

#count() → total number of rows
#describe() → mean, min, max, etc.

Total Records: 1462

Rating Summary Statistics:
+---+---+-----------------+-------------------+-----+
|min|max|             mean|             stddev|count|
+---+---+-----------------+-------------------+-----+
|2.0|5.0|4.096716821275233|0.28949702052767223| 1462|
+---+---+-----------------+-------------------+-----+



In [ ]:
#Core Analysis

#1.Ratings Distribution
# 'Score' should be 'rating'
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, FloatType

df.withColumn("rating_str", F.col("rating").cast(StringType())) \
  .groupBy("rating_str").count() \
  .orderBy(F.col("rating_str").cast(FloatType())) \
  .show()

#Shows how many users gave each rating (1–5).
#Helps understand customer satisfaction.

#2.Top Reviewed Products
# 'ProductId' should be 'product_id'
df.groupBy("product_id") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(10)

#Identifies products with highest number of reviews.

#3.Average Rating per Product
from pyspark.sql.functions import avg

# 'ProductId' should be 'product_id' and 'Score' should be 'rating'
df.groupBy("product_id") \
  .agg(avg("rating").alias("Average_Rating")) \
  .show(10)

#Calculates average rating for each product.

#4.Helpfulness Analysis
# The column 'HelpfulnessNumerator' does not exist in the dataset. This analysis will be removed.
# df.groupBy("HelpfulnessNumerator").count().show()

+----------+-----+
|rating_str|count|
+----------+-----+
|       2.0|    1|
|       2.3|    1|
|       2.6|    1|
|       2.8|    2|
|       2.9|    1|
|       3.0|    3|
|       3.1|    4|
|       3.2|    2|
|       3.3|   16|
|       3.4|   10|
|       3.5|   26|
|       3.6|   35|
|       3.7|   42|
|       3.8|   86|
|       3.9|  123|
|       4.0|  181|
|       4.1|  244|
|       4.2|  228|
|       4.3|  230|
|       4.4|  123|
+----------+-----+
only showing top 20 rows
+----------+-----+
|product_id|count|
+----------+-----+
|B09C6HXFC1|    3|
|B09CMP1SC8|    3|
|B098NS6PVG|    3|
|B08WRWPM22|    3|
|B09W5XR9RT|    3|
|B09KLVMZ3B|    3|
|B08DDRGWTJ|    3|
|B09NHVCHS9|    3|
|B085DTN6R2|    3|
|B08CF3D7QR|    3|
+----------+-----+
only showing top 10 rows
+----------+------------------+
|product_id|    Average_Rating|
+----------+------------------+
|B01M4GGIVU| 4.199999809265137|
|B07SYYVP69|3.9000000953674316|
|B0BBLHTRM9|2.9000000953674316|
|B09YHLPQYT| 4.199999809265137|
|B06

In [ ]:
#Save Cleaned Data
pandas_df = df.toPandas()
pandas_df.to_csv("cleaned_Amazon_reviews.csv", index=False)

#Converts Spark DataFrame to Pandas.
#Saves cleaned dataset for further use.